## Evaluation

### 1. Persiapan

#### 1.1. Import Library

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from config import (
  CLUSTERED_REGENCIES_CSV,
  SCALED_FEATURES_CSV,
  SELECTED_FEATURES
)

#### 1.2. Persiapan Data

In [ ]:
df_clustered = pd.read_csv(CLUSTERED_REGENCIES_CSV)
df_scaled = pd.read_csv(SCALED_FEATURES_CSV)

#### 1.3. Parameter

In [ ]:
TARGET_SILHOUETTE_MIN = 0.50

### 2. Metrik Validasi Internal Klaster

In [ ]:
scaled_cols = [f"scaled_{c}" for c in SELECTED_FEATURES if f"scaled_{c}" in df_scaled.columns]
X_scaled = df_scaled[scaled_cols].values if scaled_cols else df_scaled.values
labels = df_clustered['cluster_label'].values

sil_score = round(float(silhouette_score(X_scaled, labels)), 4)
ch_score = round(float(calinski_harabasz_score(X_scaled, labels)), 2)
db_score = round(float(davies_bouldin_score(X_scaled, labels)), 4)
num_clusters = len(np.unique(labels))

pd.DataFrame({
  'Metrik Validasi': [
    'Jumlah Klaster (K)',
    'Silhouette Coefficient',
    'Calinski-Harabasz Index',
    'Davies-Bouldin Index'
  ],
  'Nilai Evaluasi': [
    num_clusters,
    sil_score,
    ch_score,
    db_score
  ],
  'Kriteria / Target': [
    'Optimal Kneedle',
    f'>= {TARGET_SILHOUETTE_MIN} (Kerapatan Klaster)',
    'Semakin Tinggi Semakin Baik',
    'Semakin Rendah Semakin Baik'
  ]
})

### 3. Eksplorasi Visual Hasil Klasterisasi

#### 3.1. Proyeksi 2D Klaster (PCA)

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
df_pca = pd.DataFrame(X_pca, columns=['PCA1', 'PCA2'])
df_pca['cluster'] = [f"Klaster {lbl}" for lbl in labels]

plt.figure(figsize=(8, 6))
sns.scatterplot(x='PCA1', y='PCA2', hue='cluster', data=df_pca, palette='tab10', s=60, alpha=0.85)
plt.title('Proyeksi 2D Klasterisasi (Principal Component Analysis)')
plt.tight_layout()
plt.show()

#### 3.2. Distribusi Jumlah Anggota per Klaster

In [ ]:
cluster_counts = df_clustered['cluster_label'].value_counts().sort_index()

plt.figure(figsize=(8, 4.5))
sns.barplot(x=cluster_counts.index.map(lambda x: f"Klaster {x}"), y=cluster_counts.values, hue=cluster_counts.index, legend=False, palette='Blues_r')
plt.title('Distribusi Jumlah Anggota Kabupaten/Kota per Klaster')
plt.xlabel('Klaster')
plt.ylabel('Jumlah Kabupaten/Kota')
plt.tight_layout()
plt.show()

### 4. Profil Rata-Rata Fitur per Klaster

In [ ]:
active_features = [c for c in SELECTED_FEATURES if c in df_clustered.columns]
profile_df = df_clustered.groupby('cluster_label')[active_features].mean().round(2)
profile_df